## 35. 프로젝트 루트 설정

In [1]:
from pathlib import Path
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
data_dir = project_root / "data" / "raw"
print("프로젝트 루트:", project_root)
print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.exists())


프로젝트 루트: c:\dev\ai-data-analysis
데이터 폴더: c:\dev\ai-data-analysis\data\raw
데이터 폴더 존재: True


## 36. pandas와 CSV 불러오기

In [2]:
import pandas as pd
customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")


## 37. 기본 구조와 주요 키 확인


In [3]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}
for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (765, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [4]:
key_checks = {
    "customers.customer_id": customers["customer_id"],
    "products.product_id": products["product_id"],
    "orders.order_id": orders["order_id"],
    "order_items.order_item_id": order_items["order_item_id"],
}

for name, series in key_checks.items():
    print(
        name,
        "결측:", series.isna().sum(),
        "중복:", series.duplicated().sum(),

    )

customers.customer_id 결측: 0 중복: 0
products.product_id 결측: 0 중복: 0
orders.order_id 결측: 0 중복: 0
order_items.order_item_id 결측: 0 중복: 0


## 38. Series와 DataFrame 선택


In [5]:
city_series = customers["city"]
customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]
print(type(city_series))
print(type(customer_view))
display(customer_view.head())

## series가 여러개 모이면 그게 dataframe

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


## 39. 단일 조건 필터링

In [6]:

customers_over_30 = customers[
    customers["age"] >= 30
]
print(len(customers), len(customers_over_30))
display(customers_over_30.head())
## len() 함수는 데이터의 개수(길이)를 세는 함수

150 111


,customer_id,name,gender,age,city,signup_date
1,2,김정호,F,32,대구,2025-11-28
2,3,이경수,F,61,성남,2024-07-08
3,4,조영호,F,55,울산,2026-05-09
5,6,김지원,F,32,성남,2026-07-23
6,7,이상현,F,53,인천,2025-01-07


## 40. 복합 조건 필터링

In [7]:
seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]
display(seoul_over_30.head())

,customer_id,name,gender,age,city,signup_date
8,9,송지민,M,69,서울,2025-11-14
14,15,장정식,M,69,서울,2026-06-30
29,30,이민재,F,32,서울,2023-08-09
47,48,김예은,F,47,서울,2025-04-27
65,66,김재호,F,39,서울,2025-12-29


In [8]:
## 서울 또는 부산
seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"])
]
display(
    seoul_or_busan["city"].value_counts()
)
## &= 조건둘다 만족 or= 둘중에 하나만족


city
부산    16
서울    15
Name: count, dtype: int64

In [9]:
##완료 주문이 아닌 주문
not_completed = orders[
    ~(orders["order_status"] == "completed")
]
display(
    not_completed["order_status"].value_counts(
        dropna=False
    )
)
##   ~ : 부정하는거 


order_status
cancelled    64
refunded     52
Name: count, dtype: int64

## 41. 상품 가격 정렬

In [10]:
expensive_products = (
    products
    .sort_values("price", ascending=False)
    .head()
)
display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]
)
##:ascending=False: 내림차순 정렬, ascending=True: 오름차순 정렬

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000


## 42. 작업용 복사본과 파생 컬럼


In [11]:
order_items_work = order_items.copy()

# 데이터 프레임에 새로운 컬럼을 추가할 때는 아래와 같이 하면 된다.
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)

In [12]:

display(
    order_items_work[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_total",
        ]
    ].head()

)

,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000


In [13]:
# 위와 동일한 방법

print (order_items_work.head())

   order_item_id  order_id  product_id  quantity  unit_price  line_total
0              1         1         100         3      102000      306000
1              2         1          87         5       25000      125000
2              3         1           7         3      142000      426000
3              4         1           9         3      193000      579000
4              5         2          72         4      189000      756000


## 43. 수작업 검증

In [14]:
sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]
print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)
## iloc[0] : 첫번째 행을 가져오는거, iloc[1] : 두번째 행을 가져오는거

수작업: 306000
파생 컬럼: 306000
일치: True


## 44. 전체 주문상세 금액

In [17]:
all_order_amount = order_items_work["line_total"].sum()
print("전체 주문상세 금액:", all_order_amount)

전체 주문상세 금액: 257935000


## 45. 병합용 주문 컬럼 선택

In [24]:
orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]
].copy()

print(orders_for_merge.shape)
print(orders_for_merge.head())
##.copy()는 원본 데이터프레임과 완전히 별개의 새 데이터프레임을 만들라는 뜻

(300, 4)
   order_id  customer_id  order_date order_status
0         1          123  2026-06-02    completed
1         2           77  2025-08-18    cancelled
2         3          138  2025-12-15    cancelled
3         4           57  2026-02-25    cancelled
4         5          125  2026-01-16    cancelled


## 46. 주문상세와 주문 병합

In [ ]:
order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )
)


## 47. 병합 검증

In [22]:
print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))
display(
    order_sales["order_match"].value_counts(
        dropna=False
    )
)

병합 전 행 수: 765
병합 후 행 수: 765


order_match
both          764
left_only       1
right_only      0
Name: count, dtype: int64

In [25]:
# 미매칭 확인:

unmatched_orders = order_sales[
    order_sales["order_match"] != "both"
]
display(unmatched_orders.head())

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match
764,765,999,300,15,155000,2325000,NaN,NaN,NaN,left_only


## 48. 완료 주문 분석셋


In [26]:
display(
    order_sales["order_status"].value_counts(
        dropna=False
    )
)


order_status
completed    474
cancelled    162
refunded     128
NaN            1
Name: count, dtype: int64

In [27]:
completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()


In [ ]:
print("완료 주문상세 행:", len(completed_sales))
print(
    "완료 주문 수:",
    completed_sales["order_id"].nunique(),
)
print(
    "완료 주문 고객 수:",
    completed_sales["customer_id"].nunique(),
)
print(
    "완료 주문 매출:",
    completed_sales["line_total"].sum(),
)
#number of unique values:즉 고유한(중복 없는) 값의 개수를 세는 함수=nunique()

완료 주문상세 행: 474
완료 주문 수: 184
완료 주문 고객 수: 100
완료 주문 매출: 148990000


## 49. 필요한 상품 정보만 선택


In [31]:
products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]
].copy()
print(products_for_merge.head())

   product_id product_name category
0           1  전자기기 상품 001     전자기기
1           2    도서 상품 002       도서
2           3  전자기기 상품 003     전자기기
3           4  생활용품 상품 004     생활용품
4           5    식품 상품 005       식품


## 50. 완료 주문상세와 상품 병합

 


In [32]:
completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)


In [33]:
print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

474 474


product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 51. 카테고리별 매출

In [38]:
category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)
#value_counts() : 특정 컬럼의 값이 몇개씩 있는지 세는거
#  groupby()  =~별 : 특정 컬럼을 기준으로 그룹화하는거
# as_index=False : 그룹화한 컬럼을 인덱스로 만들지 말라는거
# agg() : 여러개의 집계함수를 적용하는거
# sort_values() : 정렬하는거

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
3,스포츠,31743000,85,67,295,100
5,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
1,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42
0,도서,16389000,52,46,149,58
6,패션,10587000,33,27,111,37


## 52. 카테고리 합계 검증

In [35]:
category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

148990000
148990000
True


## 53. 상품별 매출



In [39]:
product_sales = (
    completed_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        quantity_sold=("quantity", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
    )

    .sort_values("total_sales", ascending=False)
)
display(product_sales.head(10))


,product_id,product_name,category,total_sales,quantity_sold,order_count,customer_count
39,41,스포츠 상품 041,스포츠,5705000,35,12,11
11,12,식품 상품 012,식품,4375000,25,7,7
8,9,스포츠 상품 009,스포츠,3860000,20,6,5
70,72,뷰티 상품 072,뷰티,3780000,20,6,6
69,71,전자기기 상품 071,전자기기,3703000,23,5,5
66,68,스포츠 상품 068,스포츠,3640000,26,8,8
78,81,전자기기 상품 081,전자기기,3630000,22,6,6
10,11,패션 상품 011,패션,3565000,31,7,7
20,22,생활용품 상품 022,생활용품,3248000,29,8,8
86,89,생활용품 상품 089,생활용품,3090000,30,11,11


## 54. 주문 날짜 변환과 주문 월 생성


In [40]:
completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),
)

날짜 변환 실패: 0


In [41]:
completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

In [43]:
completed_items.head()

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match,product_name,category,product_match,order_month
0,1,1,100,3,102000,306000,123.0,2026-06-02,completed,both,도서 상품 100,도서,both,2026-06
1,2,1,87,5,25000,125000,123.0,2026-06-02,completed,both,도서 상품 087,도서,both,2026-06
2,3,1,7,3,142000,426000,123.0,2026-06-02,completed,both,도서 상품 007,도서,both,2026-06
3,4,1,9,3,193000,579000,123.0,2026-06-02,completed,both,스포츠 상품 009,스포츠,both,2026-06
4,13,6,83,3,24000,72000,87.0,2026-04-16,completed,both,전자기기 상품 083,전자기기,both,2026-04


## 55. 월별 매출

In [44]:
monthly_sales = (
    completed_items
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("order_month")
)

display(monthly_sales)

,order_month,total_sales,order_count,customer_count,quantity_sold
0,2025-08,7190000,10,10,63
1,2025-09,16291000,19,19,134
2,2025-10,13704000,16,16,137
3,2025-11,23360000,25,24,229
4,2025-12,7282000,9,9,73
5,2026-01,10935000,14,14,106
6,2026-02,17454000,22,20,157
7,2026-03,9589000,17,15,100
8,2026-04,14798000,16,15,151
9,2026-05,15402000,20,19,156


## 56. 고객별 구매 금액

In [45]:
customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
)
# 누가 우수고객인지 찍어보는거

In [52]:
print(customer_sales.sort_values("total_sales", ascending=False).head(10))



    customer_id  total_sales  order_count  quantity_sold
76        117.0      4100000            5             48
62        102.0      3996000            4             35
51         83.0      3880000            4             39
21         30.0      3590000            5             32
29         40.0      3523000            4             27
13         20.0      3191000            2             25
0           3.0      3178000            2             26
70        111.0      3153000            3             38
42         66.0      3093000            4             30
97        147.0      2990000            2             21


## 57. 고객 속성 연결

In [53]:
customer_attributes = customers[
    ["customer_id", "gender", "age", "city"]
].copy()


In [54]:
customer_sales_detail = (
    customer_sales
    .merge(
        customer_attributes,
        on="customer_id",
        how="left",
        validate="one_to_one",
        indicator="customer_match",
    )
    .sort_values("total_sales", ascending=False)
)

In [55]:
display(
    customer_sales_detail["customer_match"].value_counts(
        dropna=False
    )
)
display(customer_sales_detail.head(10))


customer_match
both          100
left_only       0
right_only      0
Name: count, dtype: int64

,customer_id,total_sales,order_count,quantity_sold,gender,age,city,customer_match
76,117.0,4100000,5,48,F,65,성남,both
62,102.0,3996000,4,35,M,60,고양,both
51,83.0,3880000,4,39,F,22,수원,both
21,30.0,3590000,5,32,F,32,서울,both
29,40.0,3523000,4,27,M,23,서울,both
13,20.0,3191000,2,25,F,20,인천,both
0,3.0,3178000,2,26,F,61,성남,both
70,111.0,3153000,3,38,F,41,광주,both
42,66.0,3093000,4,30,F,39,서울,both
97,147.0,2990000,2,21,M,19,부산,both


##  결과 폴더 생성 

In [56]:
output_dir = project_root / "reports" / "chapter04"
output_dir.mkdir(parents=True, exist_ok=True)
print(output_dir)

c:\dev\ai-data-analysis\reports\chapter04


## 59. 결과 파일 저장

In [57]:
outputs = {
    "category_sales.csv": category_sales,
    "product_sales.csv": product_sales,
    "monthly_sales.csv": monthly_sales,
    "customer_sales.csv": customer_sales_detail,
}
for file_name, df in outputs.items():
    output_path = output_dir / file_name
    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(
        file_name,
        output_path.exists(),
        output_path.stat().st_size,
    )

category_sales.csv True 309
product_sales.csv True 4833
monthly_sales.csv True 392
customer_sales.csv True 3648


## 60. 저장 결과 다시 읽기

In [58]:
saved_category_sales = pd.read_csv(
    output_dir / "category_sales.csv"
)
display(saved_category_sales.head())
print(saved_category_sales.shape)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
0,스포츠,31743000,85,67,295,100
1,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
3,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42


(7, 6)


## 61. 병합 점검 함수

In [59]:
def check_merge_result(
    *,
    name: str,
    left_rows: int,
    merged: pd.DataFrame,
    indicator_column: str,
) -> None:
    print(f"[{name}]")
    print("병합 전 행 수:", left_rows)
    print("병합 후 행 수:", len(merged))
    print(
        merged[indicator_column].value_counts(
            dropna=False
        )
    )


In [60]:
# 위에 만든 함수 호출
check_merge_result(
    name="주문상세-주문",
    left_rows=len(order_items_work),
    merged=order_sales,
    indicator_column="order_match",
)

[주문상세-주문]
병합 전 행 수: 765
병합 후 행 수: 765
order_match
both          764
left_only       1
right_only      0
Name: count, dtype: int64


## 62. 집계 합계 검증 함수

In [61]:
def check_total(
    *,
    name: str,
    source_total: float,
    summary_total: float,
) -> None:
    difference = source_total - summary_total
    print(f"[{name}]")
    print("원본 합계:", source_total)
    print("요약 합계:", summary_total)
    print("차이:", difference)

In [62]:
# 위에 만든 함수 호출
check_total(
    name="카테고리별 매출",
    source_total=completed_items["line_total"].sum(),
    summary_total=category_sales["total_sales"].sum(),
)

[카테고리별 매출]
원본 합계: 148990000
요약 합계: 148990000
차이: 0


## 63 LLM 코드 검증표

| 검증 항목 | 확인 내용 | 결과 |
|---|---|---|
| DataFrame | 실제 변수명과 같은가? | orders, order_items, products, customers 네 개의 실제 변수명을 그대로 사용했고, 다른 이름으로 바꾸거나 임의의 변수를 만들지 않았음 |
| 컬럼 | 실제 컬럼만 사용하는가? | 45번(orders_for_merge), 49번(products_for_merge), 57번(customer_attributes)에서 필요한 컬럼만 명시적으로 골라 사용했고, 원본에 없는 컬럼을 새로 만들어 쓴 곳은 없음 |
| 상태값 | completed 표기가 맞는가? | 48번에서 필터링하기 전에 order_status의 value_counts()로 실제 값 목록을 먼저 확인했고, 그 결과에 있는 "completed" 표기를 그대로 사용해 필터링함 |
| 계산식 | quantity × unit_price인가? | 42번에서 line_total = quantity × unit_price로 정의했고, 43번에서 iloc[0]으로 뽑은 한 행을 손으로 직접 곱해본 값과 파생 컬럼 값을 비교해 일치함을 확인함 |
| 분석 범위 | 완료 주문만 포함하는가? | 48번에서 order_status == "completed"로 필터링한 completed_sales/completed_items를 만들었고, 이후 51~57번의 모든 집계가 이 데이터만 사용함 |
| 주문 수 | nunique()를 사용하는가? | 48, 51, 53, 55, 56번 전부 order_id.nunique()로 계산했고, 행 개수(len)를 주문 수로 오인해 쓴 곳은 없음 |
| 병합 키 | 실제 관계와 맞는가? | order_id로 order_items와 orders를 연결(46번), product_id로 order_items와 products를 연결(50번), customer_id로 고객별 매출과 고객 속성을 연결(57번) — 셋 다 실제 데이터 관계와 일치함 |
| validate | many_to_one이 적용되었는가? | order 병합(46번)과 product 병합(50번)은 many_to_one을, customer 병합(57번)은 one_to_one을 적용함. customer 병합은 양쪽 다 고객당 1행이라 one_to_one이 맞는 선택임 |
| indicator | 미매칭을 확인하는가? | 세 병합 모두 indicator 컬럼(order_match, product_match, customer_match)을 부여함. 실제 실행 결과 order_match에서만 1건 미매칭(left_only)이 나왔고, order_id=999라는 실제로 orders 테이블에 존재하지 않는 주문번호가 원인이었음. 이 1건은 order_status가 NaN이 되어 48번 completed 필터에서 자동으로 빠지므로 최종 매출 집계에는 영향이 없음을 확인함. product_match, customer_match는 전부 both로 미매칭 없음 |
| 행 수 | 병합 전후를 비교하는가? | 47번에서 order 병합 전후 행 수(765건 → 765건, 동일)를 명시적으로 출력해 병합으로 행이 늘거나 줄지 않았음을 확인함. 50번과 57번도 병합 전후 길이를 출력하긴 하나 "병합 전/후" 라벨 없이 단순 출력이라 가독성 면에서 47번만큼 명확하지는 않음 |
| 합계 | 원본과 요약 합계를 비교하는가? | 52번에서 category_sales의 total_sales 합계와 completed_items의 line_total 합계를 비교해 True(일치)를 확인했고, 62번에서 이를 재사용 가능한 함수(check_total)로 만들어 같은 결과를 재확인함 |
| 개인정보 | 원본 고객 정보를 요구하지 않는가? | 57번에서 customer_attributes에 gender, age, city만 포함시켰고, 이름·연락처·주소 등 실제 개인 식별 정보는 사용하지 않음 |

## 64. 실행 성공과 분석 타당성 구분

| 항목 | 확인 내용 | 결과 |
|---|---|---|
| 주문 상태 | 취소 주문이 포함되지 않았는가? | 48번에서 order_status == "completed"로만 필터링했고, 필터링 전 value_counts()로 completed 외 다른 상태값들이 존재함을 확인한 뒤 그 값들을 제외시켰으므로 취소 등 다른 상태 주문은 최종 집계에 포함되지 않음 |
| 주문 수 계산 | 주문상세 행 수를 주문 수로 계산하지 않았는가? | order_count는 항상 order_id.nunique()로 계산했고, order_item 단위의 행 개수(len)를 그대로 주문 수로 쓴 곳은 없음 |
| 단가 기준 | 상품 마스터 가격을 실제 판매 단가로 사용하지 않았는가? | 49번의 products_for_merge에는 product_id, product_name, category만 포함시키고 products의 price 컬럼은 아예 가져오지 않았음. line_total 계산에는 order_items의 unit_price(판매 시점 단가)만 사용함 |
| 병합 행 수 | 병합으로 인해 행이 증가하지 않았는가? | 46·50·57번 모두 validate로 관계를 사전에 강제했고, 47번에서 병합 전후 행 수가 765건으로 동일함을 실제로 확인함. many_to_one/one_to_one 위반 시 에러가 나므로, 에러 없이 실행됐다는 것 자체가 행 증가가 없었다는 근거이기도 함 |
| 미매칭 처리 | 미매칭 행이 누락되지 않았는가? | order_id=999인 order_item 1건이 orders에 존재하지 않아 미매칭됨을 indicator로 확인했고, 원인(존재하지 않는 주문번호)까지 파악함. 이 행은 조용히 사라진 게 아니라 존재를 확인했고, order_status가 NaN이 되어 completed 필터에서 자동 제외되므로 최종 매출에는 영향이 없음이 검증됨 |
| 날짜 타입 | 날짜가 문자열 상태로 남아있지 않은가? | 54번에서 pd.to_datetime(errors="coerce")로 변환했고, 변환 실패 건수(NaT 개수)를 출력해 0건임을 확인함. order_month도 to_period("M")으로 파생시켜 월별 집계(55번)에 문제없이 사용됨 |
| 그룹바이 기준 | 그룹바이 기준이 질문(카테고리)과 일치하는가? | 51번에서 groupby("category")로 집계했고, 이는 분석 목표("완료 주문 기준 카테고리별 매출")와 정확히 일치함 |
| 결과 해석 | 결과를 과도하게 해석하지 않았는가? | category_sales와 product_sales의 order_count를 각각 합산하면 전체 완료 주문 수(184건)보다 커지는데(예: 카테고리 합산 396건), 이는 한 주문이 여러 카테고리 상품을 포함할 수 있어서 생기는 정상적인 현상이며 오류가 아님을 확인함. 다만 이 설명이 노트북 안에 주석으로 남아있지는 않아, 나중에 이 결과만 보는 사람이 오해하지 않도록 주석을 추가해두는 게 좋음 |